# Database Design — Low-Level Design (LLD)

**System:** File Validation & Routing Framework (`fvr` database, MongoDB) • **Version:** 1.0 • **Status:** DRAFT FOR REVIEW • **Companion HLD:** `Database Design HLD — File Validation & Routing (MongoDB)`

## 1. Purpose

Field-level specification of all 21 collections + GridFS: document schemas (field, type, required, constraints), indexes, relationships, write/read patterns, the state machine transition table, enum dictionary, seed data, and operational configuration. Ready-to-run `$jsonSchema` validators and `createIndex` DDL live in the master design (`MongoDB Index & Schema Validation Specs` cell) — this LLD is the authoritative field reference those DDLs implement.

## 2. Conventions

* **IDs:** prefixed strings, unique per collection: `usr_`, `src_`, `ft_`, `sch_`, `rule_`, `pol_`, `vol_`, `ret_`, `evt_`, `ses_`, `file_`, `vr_`, `bat_`, `trf_`, `aud_`, `flc_`, `not_`, `inj_`, `rr_`, `rec_`.
* **Timestamps:** BSON dates, UTC, always.
* **Enums:** single source of truth = the enum dictionary (cell 7); `$jsonSchema` validators enforce them at write time.
* **Deletes:** hard deletes only where explicitly specified (`fs.files`/`fs.chunks` purge, `event_log` TTL); everything else is status/flag-driven (`is_active`, `status`).

## 3. Relationship Map (application-enforced)

```
upload_sessions ──1:N── file_registry ──1:N── validation_results
       │                   │    │        ├──1:N── file_transfers
       │                   │    │        ├──1:N── file_lifecycle
       │                   │    │        ├──1:N── databricks_ingestion_results (Ph3)
       │                   │    │        └──1:N── reconciliation_results (Ph3)
       │                   │    └─ N:1 ── file_types ──1:N── schema_versions
       │                   │                ├──1:N── validation_rules (applies_to)
       │                   │                └──1:1── volume_map
users ──1:N── upload_sessions     source_systems ──1:N── file_registry / event_log
event_log.correlation_id ── ties every artifact of a file's journey together
retention_policies ──drive── archive_jobs ──append── file_lifecycle
```

MongoDB has no foreign-key constraints — referential integrity is enforced by the FastAPI write layer + `$jsonSchema` patterns, and verified by reconciliation (Phase 3).

## 4. Registry Layer — 8 Collections

### 4.1 `users` — portal identity + RBAC

| Field | Type | Req | Constraints | Description |
| --- | --- | --- | --- | --- |
| `user_id` | string | Y | `^usr_` | PK |
| `email` | string | Y | unique index | identity (SSO claim) |
| `display_name` | string | Y | — | UI label |
| `roles` | array\<string\> | Y | enum ROLES | `UPLOADER`, `OPS`, `ADMIN` — gates API + React routes |
| `department` | string | N | — | reporting dimension |
| `is_active` | bool | Y | default true | soft delete (never hard-delete a user — audit references them) |
| `created_at` / `last_login_at` | date | Y / N | — | lifecycle |

**W:** SSO login handler (upsert by email). **R:** auth middleware, admin screens.

### 4.2 `source_systems` — source registry

| Field | Type | Req | Constraints | Description |
| --- | --- | --- | --- | --- |
| `source_id` | string | Y | `^src_`, unique | PK |
| `name` | string | Y | — | display name |
| `source_type` | string | Y | enum SOURCE_TYPE | PORTAL / SHAREPOINT / SFTP / DB_VOLUME |
| `connection_ref` | string | Y | — | **vault reference only — never credentials** |
| `poll_or_push` | string | Y | enum PUSH / POLL | detection model |
| `detection_config` | object | N | — | watcher interval / webhook path / watermark cursor |
| `is_active` | bool | Y | — | soft disable |

**W:** admin CRUD (audited). **R:** event adapters. **Seed:** `src_portal` (PUSH).

### 4.3 `file_types` — file contracts

| Field | Type | Req | Constraints | Description |
| --- | --- | --- | --- | --- |
| `file_type_id` | string | Y | `^ft_`, unique | PK |
| `code` | string | Y | unique | `customer`, `order`, … matches volume subfolder |
| `description` | string | N | — | — |
| `allowed_extensions` | array\<string\> | Y | minItems 1 | `.csv`, `.xlsx` |
| `max_file_size_mb` / `min_rows` | int | N | — | file-level bounds |
| `encoding` / `delimiter` | string | N | — | parse config |
| `header_row` | bool | Y | — | header expected |
| `identification` | object | N | — | `{patterns: [regex], header_fingerprint: [names]}` — Phase-2 identification for unannounced sources |
| `is_active` | bool / `version` | int | Y | — | soft lifecycle |

### 4.4 `schema_versions` — versioned field contracts

| Field | Type | Req | Constraints | Description |
| --- | --- | --- | --- | --- |
| `schema_id` | string | Y | `^sch_`, unique | PK |
| `file_type_id` | string | Y | ref `file_types` | owner |
| `version` | int | Y | unique per `(file_type_id, version)` | monotonic |
| `status` | string | Y | enum SCHEMA_STATUS | draft / active / retired — **exactly one active per file type** (app-enforced) |
| `effective_from` | date | Y | — | which version validates a file at arrival time |
| `fields` | array\<object\> | Y | minItems 1, ordered | `{order, name, data_type, nullable, max_length, format, is_pii, pii_classification}` — `order` drives COLUMN_ORDER_MATCH |
| `created_by` / `created_at` | string / date | Y | — | governance |

### 4.5 `validation_rules` — extensible rule registry

| Field | Type | Req | Constraints | Description |
| --- | --- | --- | --- | --- |
| `rule_id` | string | Y | `^rule_`, unique | PK |
| `rule_code` | string | Y | unique per `(rule_code, version)` | resolves to a registered validator plugin |
| `category` | string | Y | enum RULE_CATEGORY | FILE / HEADER / CONTENT — execution phase |
| `severity` | string | Y | enum RULE_SEVERITY | ERROR (quarantine) / WARN (shadow) |
| `applies_to` | string | N | ref `file_types` or null | null = global |
| `params` | object | Y | free-form | `{"max_mb": 500}`, `{"allowed_regions": ["EU"]}` … |
| `priority` | int | Y | — | execution order within category |
| `is_active` / `version` / `created_by` / `updated_at` | mixed | Y | — | lifecycle + governance |

**W:** admin CRUD (audited, approver recorded). **R:** rule engine at validation time.

### 4.6 `routing_policies` — first-match by priority

| Field | Type | Req | Constraints | Description |
| --- | --- | --- | --- | --- |
| `policy_id` | string | Y | `^pol_`, unique | PK |
| `name` | string | Y | — | display |
| `priority` | int | Y | — | lower = evaluated first |
| `match` | object | Y | — | condition tree, e.g. `{any_error: true}`, `{status: "PASSED"}` |
| `action` | string | Y | enum ROUTING_ACTION | MOVE_INTAKE / MOVE_QUARANTINE / REJECT / NOTIFY_ONLY |
| `is_active` | bool | Y | — | — |

### 4.7 `volume_map` — destination resolution

| Field | Type | Req | Constraints | Description |
| --- | --- | --- | --- | --- |
| `map_id` | string | Y | `^vol_`, unique | PK |
| `file_type_id` | string | Y | `ft_id` or `*` | `*` = all types |
| `destination_type` | string | Y | enum DESTINATION_TYPE | INTAKE / QUARANTINE / REJECT |
| `volume_root` | string | Y | — | `/Volumes/<cat>/<schema>/intake` |
| `subfolder` | string | Y | — | defaults to file type code |
| `naming_convention` | string | Y | default `{file_type}_{file_id}_{ts}.ext` | bronze attribution without joins |

### 4.8 `retention_policies` (Phase 3)

| Field | Type | Req | Constraints | Description |
| --- | --- | --- | --- | --- |
| `policy_id` | string | Y | `^ret_`, unique | PK |
| `file_type_id` | string | Y | or `*` | scope |
| `status_class` | string | Y | enum STATUS_CLASS | INTAKE_COPY / QUARANTINED / UNIDENTIFIED / ALL |
| `retention_days` / `archive_after_days` / `purge_after_days` | int | Y | — | lifecycle timers (purge additionally requires MATCHED) |
| `archive_target` | string | N | — | archive volume path |
| `legal_hold` | bool | Y | default false | **blocks purge absolutely** |

## 5. Eventing + Execution Layer — 6 Collections

### 5.1 `event_log` — canonical event store + queue

| Field | Type | Req | Constraints | Description |
| --- | --- | --- | --- | --- |
| `event_id` | string | Y | `^evt_`, unique | PK |
| `event_type` | string | Y | enum EVENT_TYPE | 12 types (cell 7) |
| `source_id` | string | Y | ref `source_systems` | origin |
| `detection_method` | string | Y | enum DETECTION_METHOD | PUSH / WEBHOOK / WATCHER / POLL |
| `dedup_key` | string | Y | **unique `(event_type, dedup_key)`** | portal: `session_id\|filename\|part_index`; others: `source\|uri\|mtime` |
| `payload` | object | Y | — | event body (uri, size, mtime, actor…) |
| `processing_status` | string | Y | enum EVENT_STATUS | RECEIVED → PROCESSING → PROCESSED / DEAD |
| `retry_count` / `last_error` | int / string | Y / N | — | retry telemetry |
| `correlation_id` | string | N | — | joins the whole file journey |
| `received_at` | date | Y | — | arrival |

**W:** adapters (insert only) + worker (status updates). **R:** change-stream worker; DEAD re-drive view. **Index note:** partial index on DEAD; TTL on PROCESSED only (proposed 30 d).

### 5.2 `upload_sessions` — portal interaction record

| Field | Type | Req | Constraints | Description |
| --- | --- | --- | --- | --- |
| `session_id` | string | Y | `^ses_`, unique | PK |
| `source_id` | string | Y | — | always `src_portal` in Phase 1 |
| `user_id` / `user_email` | string | Y | ref `users` | identity |
| `client_ip` / `machine_name` / `user_agent` | string | Y / N / N | — | **upload context capture** |
| `auth_method` | string | N | — | e.g. SSO_OIDC |
| `roles_snapshot` | array\<string\> | Y | enum ROLES | frozen at upload time (audit-safe) |
| `files_declared_count` | int | N | — | from the manifest — enables completeness check |
| `files_received_count` | int | Y | — | actual |
| `parts` | array\<object\> | Y | — | `{file_id, gridfs_file_id, status, chunks_received}` — resumable-upload tracking |
| `started_at` / `completed_at` | date | Y / N | — | session timing |

### 5.3 `file_registry` — control entity + state machine

| Field | Type | Req | Constraints | Description |
| --- | --- | --- | --- | --- |
| `file_id` | string | Y | `^file_`, unique | PK — the FK spine of the whole model |
| `source_id` | string | Y | ref `source_systems` | origin |
| `original_name` | string | Y | — | as uploaded |
| `staged_path` | string | N | — | for non-GridFS staging (Phase-2 pollers) |
| `gridfs_file_id` | objectId | Y | ref `fs.files._id` | bytes pointer |
| `size_bytes` | int/long | Y | ≥ 0 | — |
| `extension` / `mime_type` | string | Y / N | — | — |
| `sha256_checksum` | string | Y | **unique**, 64 hex chars | duplicate detection key |
| `file_type_id` / `schema_version_id` | string | N | ref | identified contract |
| `identification_method` | string | N | enum ID_METHOD | EXPLICIT / FILENAME_PATTERN / HEADER_FINGERPRINT / CONTENT_SNIFF |
| `identification_confidence` / `identification_evidence` | number / object | N | — | how the type was resolved |
| `unidentified_reason` | string | N | — | when nothing matched |
| `upload_session_id` / `uploaded_by` / `client_ip` / `machine_name` | mixed | N | — | upload context (denormalized from session for single-doc ops queries) |
| `current_status` | string | Y | enum FILE_STATUS (cell 7) | state machine |
| `batch_id` | string | N | ref `batches` | Phase-2 grouping |
| `retry_count` | int | Y | — | FRAMEWORK_ERROR retries |
| `received_at` / `updated_at` | date | Y | — | lifecycle |

**W:** upload API + pipeline worker (guarded transitions only). **R:** every screen; indexes drive the ops board.

### 5.4 `validation_results` — rule outcomes with evidence

| Field | Type | Req | Constraints | Description |
| --- | --- | --- | --- | --- |
| `result_id` | string | Y | `^vr_`, unique | PK |
| `file_id` | string | Y | ref `file_registry`, indexed | — |
| `batch_id` | string | N | — | — |
| `rule_id` / `rule_version` | string / int | Y | ref | **exact rule version that ran** |
| `status` | string | Y | enum RULE_RESULT | PASS / FAIL / WARN / SKIPPED / ERROR |
| `details` | object | Y | — | evidence: mismatched columns, offending values, regions found |
| `executed_at` / `duration_ms` | date / int | Y | — | telemetry |

### 5.5 `batches` (Phase 2) — run control + watermarks

| Field | Type | Req | Constraints | Description |
| --- | --- | --- | --- | --- |
| `batch_id` | string | Y | `^bat_`, unique | PK |
| `source_id` | string | Y | — | — |
| `triggered_by` | string | Y | enum TRIGGER | event / schedule / user |
| `watermark` | object | N | — | incremental cursor (SFTP mtime, SharePoint change token) |
| `status` / counters | mixed | Y | — | files_total / passed / failed / quarantined |
| `started_at` / `ended_at` | date | Y/N | — | — |

### 5.6 `file_transfers` — routing manifest

| Field | Type | Req | Constraints | Description |
| --- | --- | --- | --- | --- |
| `transfer_id` | string | Y | `^trf_`, unique | PK |
| `file_id` | string | Y | ref `file_registry` | — |
| `policy_id` | string | N | ref `routing_policies` | decision link |
| `from_location` | string | Y | `gridfs:<objectId>` | source |
| `to_location` | string | Y | resolved volume path | incl. subfolder + renamed filename |
| `direction` | string | Y | enum TRANSFER_DIRECTION | MONGO_TO_INTAKE / MONGO_TO_QUARANTINE / MONGO_TO_ARCHIVE |
| `status` | string | Y | enum TRANSFER_STATUS | PENDING / IN_PROGRESS / COMPLETED / FAILED / RETRYING |
| `attempts` / `last_error` | int / string | Y / N | attempts ≥ 0 | retry handling |
| `source_sha256` / `dest_sha256` | string | Y / N | 64 hex | **post-copy checksum verification** |
| `bytes_written` / `worker_id` | mixed | N | — | telemetry |
| `started_at` / `completed_at` | date | N | — | — |

**Idempotency:** partial-unique `(file_id, direction)` where `status: COMPLETED` — a file can never land in Intake twice.

## 6. Governance Layer — 4 Collections

### 6.1 `audit_log` — append-only trail

| Field | Type | Req | Constraints | Description |
| --- | --- | --- | --- | --- |
| `event_id` | string | Y | `^aud_`, unique | PK |
| `ts` | date | Y | — | — |
| `actor` | string | Y | user_id or `system:<component>` | who |
| `entity_type` / `entity_id` | string | Y | — | what (file / rule / session / config) |
| `action` | string | Y | enum AUDIT_ACTION | CONFIG_CHANGE / STATUS_CHANGE / FILE_ROUTED / RULE_ADDED / OTHER |
| `before` / `after` | object | N | — | diff |
| `approver` | string | N | — | required on config changes (ADMIN dual control) |
| `correlation_id` | string | N | — | journey tracing |

**Enforcement:** the app user holds **insert-only** on this collection — updates/deletes are impossible at the DB level.

### 6.2 `file_lifecycle` — physical location history

| Field | Type | Req | Constraints | Description |
| --- | --- | --- | --- | --- |
| `event_id` | string | Y | `^flc_`, unique | PK |
| `file_id` | string | Y | ref `file_registry`, indexed | — |
| `location` | string | Y | enum LOCATION | GRIDFS / INTAKE_VOLUME / QUARANTINE / ARCHIVE / PURGED |
| `timestamp` | date | Y | — | when it physically moved |
| `triggered_by_job` | string | N | ref `archive_jobs` | — |

### 6.3 `notifications` — alert routing config

| Field | Type | Req | Constraints | Description |
| --- | --- | --- | --- | --- |
| `notification_id` | string | Y | `^not_`, unique | PK |
| `event` | string | Y | enum EVENT_TYPE + ops events | trigger |
| `condition` | object | N | — | e.g. `{severity: {"$gte": 2}}`, `{stuck_hours: {"$gt": 24}}` |
| `channel` | string | Y | enum CHANNEL | email / teams / webhook |
| `recipients` | array\<string\> | Y | — | — |
| `is_active` | bool | Y | — | — |

### 6.4 `archive_jobs` (Phase 3) — retention run control

| Field | Type | Req | Constraints | Description |
| --- | --- | --- | --- | --- |
| `job_id` | string | Y | `^arj_`, unique | PK |
| `policy_id` | string | Y | ref `retention_policies` | — |
| `status` | string | Y | enum JOB_STATUS | RUNNING / COMPLETED / FAILED |
| `files_scanned` / `files_archived` / `files_purged` | int | Y | — | counters |
| `started_at` / `ended_at` | date | Y / N | — | — |

## 7. Resilience Layer — 3 Collections (Phase 3)

### 7.1 `databricks_ingestion_results` — pushback target

| Field | Type | Req | Constraints | Description |
| --- | --- | --- | --- | --- |
| `ingestion_id` | string | Y | `^inj_`, **unique** | `<file_id>\|<bronze_table>\|<run_id>` — idempotent upsert key |
| `run_id` | string | Y | indexed | Databricks run |
| `bronze_table` | string | Y | — | fully qualified |
| `file_type_id` / `file_id` / `sha256` / `filename` | string | Y | ref | attribution — filename embeds `file_id` by naming convention |
| `rows_read` / `rows_written` / `rows_quarantined_silver` | int/long | Y | — | row accounting |
| `dq_results` | object | N | — | silver DQ summary `{rules_failed, quarantined_rows, silver_status}` |
| `processed_at` | date | Y | — | — |

**W:** FastAPI internal endpoint (service principal, allowlisted) — the only non-portal writer.

### 7.2 `reconciliation_runs` / 7.3 `reconciliation_results`

`reconciliation_runs`: `run_id` (PK, `^rr_`), `scope` (enum: batch / daily / file_type), `expected_files`, `matched`, `discrepancies`, `status` (enum JOB_STATUS), `started_at`, `ended_at`.

`reconciliation_results` — one doc per file per run:

| Field | Type | Req | Constraints | Description |
| --- | --- | --- | --- | --- |
| `result_id` | string | Y | `^rec_`, unique | PK |
| `file_id` / `run_id` | string | Y | **unique `(file_id, run_id)`** | — |
| `uploaded_flag` / `routed_flag` / `transfer_sha_match` / `bronze_processed_flag` | bool | Y | — | chain evidence |
| `row_delta` | int | N | — | file rows vs bronze rows |
| `status` | string | Y | enum RECON_STATUS | MATCHED / MISSING_IN_BRONZE / ROW_MISMATCH / DUPLICATE_PROCESSED / STUCK_IN_TRANSIT |
| `severity` / `resolution_status` | string | N | — | auto-classification → alert → track to closure |

**Purge gate:** `archive_jobs` deletes GridFS objects only where a `MATCHED` result exists (or legal-hold override) — enforced at the application layer, audited in `file_lifecycle`.

## 8. GridFS Design (LLD)

### 8.1 Bucket configuration

| Item | Value | Notes |
| --- | --- | --- |
| Bucket | `fs` (default: `fs.files`, `fs.chunks`) | single bucket for ALL file types (HLD decision D2) |
| `chunkSizeBytes` | 261144 (255 KB) | raise to 1 MB if typical files > ~100 MB |
| `fs.files` index | default unique `(filename, uploadDate)` | — |
| `fs.chunks` index | default unique `(files_id, n)` | — |
| Secondary index | `{"metadata.file_id": 1}` | chunk-level → registry reverse lookup |

### 8.2 `fs.files` metadata contract (minimal, reverse-lookup only)

```json
{
  "_id": ObjectId("665f1a2e..."),
  "filename": "ses_01HV4B.../customer_master_20261109.csv",
  "chunkSize": 261144,
  "uploadDate": {"$date": "2026-11-09T09:14:10Z"},
  "length": 10485760,
  "md5": "...",
  "metadata": {
    "file_id": "file_01HV4C...",
    "sha256": "9f86d081...",
    "original_name": "customer_master_20261109.csv",
    "upload_session_id": "ses_01HV4B..."
  }
}
```

`filename` is namespaced `session_id/original_name` for uniqueness; the framework never queries GridFS for state — always `file_registry` → `gridfs_file_id`.

### 8.3 Upload / resume protocol (portal)

1. `POST /api/uploads/manifest` → `upload_sessions` doc with `files_declared_count`.
2. Chunks streamed per part; `upload_sessions.parts[].chunks_received` updated per chunk (idempotent counter — resume continues from last received).
3. On part completion: sha256 computed stream-side, `fs.files` finalized with metadata; `file_registry` doc created (RECEIVED).
4. `complete` call: session counters finalized, `FILE_ARRIVED` event emitted; pipeline takes over.

### 8.4 Purge protocol (Phase 3)

`archive_jobs` deletes `fs.files` + `fs.chunks` by `_id` **only after**: `MATCHED` reconciliation result **and** `purge_after_days` elapsed **and** no `legal_hold`. Each purge appends `file_lifecycle {location: PURGED}` and an `audit_log` entry. GridFS copies are otherwise retained indefinitely in Phase 1.

## 9. State Machine & Enum Dictionary

### 9.1 `file_registry` transitions (illegal transitions rejected + audited)

| From | To | Guard / trigger | Actor |
| --- | --- | --- | --- |
| RECEIVED | IDENTIFYING | file staged + event processed | worker |
| IDENTIFYING | VALIDATING | type resolved | worker |
| IDENTIFYING | UNIDENTIFIED | no identification method matched | worker |
| UNIDENTIFIED | VALIDATING | manual re-classification | OPS user (audited) |
| VALIDATING | PASSED / WARNED / FAILED | all rules executed; severity aggregation | worker |
| PASSED, WARNED | ROUTED_INTAKE | policy MOVE_INTAKE + transfer COMPLETED | transfer worker |
| FAILED, UNIDENTIFIED | ROUTED_QUARANTINE | policy MOVE_QUARANTINE + transfer COMPLETED | transfer worker |
| any | FRAMEWORK_ERROR | infrastructure failure | worker → auto-retry (max 3) |
| FRAMEWORK_ERROR | previous state | retry success | worker |
| any (policy) | REJECTED | explicit REJECT action | admin (audited) |

### 9.2 Enum dictionary (single source of truth — mirrored in `$jsonSchema`)

| Enum | Values |
| --- | --- |
| ROLES | `UPLOADER`, `OPS`, `ADMIN` |
| SOURCE_TYPE | `PORTAL`, `SHAREPOINT`, `SFTP`, `DB_VOLUME` |
| SCHEMA_STATUS | `draft`, `active`, `retired` |
| RULE_CATEGORY | `FILE`, `HEADER`, `CONTENT` |
| RULE_SEVERITY | `ERROR`, `WARN` |
| RULE_RESULT | `PASS`, `FAIL`, `WARN`, `SKIPPED`, `ERROR` |
| ROUTING_ACTION | `MOVE_INTAKE`, `MOVE_QUARANTINE`, `REJECT`, `NOTIFY_ONLY` |
| DESTINATION_TYPE | `INTAKE`, `QUARANTINE`, `REJECT` |
| EVENT_TYPE | `FILE_ARRIVED`, `FILE_STAGED`, `FILE_IDENTIFIED`, `FILE_UNIDENTIFIED`, `VALIDATION_COMPLETED`, `FILE_ROUTED`, `TRANSFER_COMPLETED`, `TRANSFER_FAILED`, `BRONZE_PROCESSED`, `RECONCILIATION_COMPLETED`, `FILE_ARCHIVED`, `FILE_PURGED` |
| DETECTION_METHOD | `PUSH`, `WEBHOOK`, `WATCHER`, `POLL` |
| EVENT_STATUS | `RECEIVED`, `PROCESSING`, `PROCESSED`, `DEAD` |
| FILE_STATUS | `RECEIVED`, `IDENTIFYING`, `VALIDATING`, `PASSED`, `FAILED`, `WARNED`, `UNIDENTIFIED`, `ROUTED_INTAKE`, `ROUTED_QUARANTINE`, `REJECTED`, `FRAMEWORK_ERROR` |
| ID_METHOD | `EXPLICIT`, `FILENAME_PATTERN`, `HEADER_FINGERPRINT`, `CONTENT_SNIFF` |
| TRANSFER_DIRECTION | `MONGO_TO_INTAKE`, `MONGO_TO_QUARANTINE`, `MONGO_TO_ARCHIVE` |
| TRANSFER_STATUS | `PENDING`, `IN_PROGRESS`, `COMPLETED`, `FAILED`, `RETRYING` |
| LOCATION | `GRIDFS`, `INTAKE_VOLUME`, `QUARANTINE`, `ARCHIVE`, `PURGED` |
| AUDIT_ACTION | `CONFIG_CHANGE`, `STATUS_CHANGE`, `FILE_ROUTED`, `RULE_ADDED`, `OTHER` |
| CHANNEL | `email`, `teams`, `webhook` |
| TRIGGER | `event`, `schedule`, `user` |
| STATUS_CLASS | `INTAKE_COPY`, `QUARANTINED`, `UNIDENTIFIED`, `ALL` |
| RECON_STATUS | `MATCHED`, `MISSING_IN_BRONZE`, `ROW_MISMATCH`, `DUPLICATE_PROCESSED`, `STUCK_IN_TRANSIT` |
| JOB_STATUS | `RUNNING`, `COMPLETED`, `FAILED` |

**Adding an enum value** = LLD + `$jsonSchema` validator update + release note — validators reject unknown values, so old app versions fail safe on new data.

## 10. Query Patterns, Index Summary & Operations

### 10.1 Query patterns → indexes

| Screen / job | Query | Index used |
| --- | --- | --- |
| Ops status board | `file_registry` by `current_status` + time | `{current_status: 1}` + `{source_id, received_at: -1}` |
| My uploads | `file_registry` by `upload_session_id`; sessions by user | `{upload_session_id: 1}` + `{user_id, started_at: -1}` |
| Results screen | `validation_results` by `file_id` | `{file_id: 1, executed_at: -1}` |
| Transfer retries | `file_transfers` by `status` | `{status: 1, created_at: 1}` |
| DEAD re-drive | `event_log` DEAD view | partial `{received_at: 1} where DEAD` |
| Duplicate check | `file_registry` by `sha256_checksum` | unique `{sha256_checksum: 1}` |
| Reconciliation | join `file_registry` ↔ `databricks_ingestion_results` by `file_id` | `{file_id: 1, processed_at: -1}` |
| Audit drill-down | `audit_log` by entity | `{entity_id: 1, ts: -1}` |

Full `createIndex` DDL: master design, `MongoDB Index & Schema Validation Specs` (unchanged, authoritative).

### 10.2 Seed data (Step 1 of the build sequence)

* `source_systems`: `src_portal`
* `file_types`: customer, order, billing, product, employee (+ `volume_map` rows incl. `*`→QUARANTINE)
* `schema_versions`: one active per file type
* `validation_rules`: the 8 Phase-1 seeds (FILE_TYPE_ALLOWED, FILE_SIZE_MAX, FILE_NOT_EMPTY, FILE_NOT_DUPLICATE, HEADER_REQUIRED, SCHEMA_MATCH, COLUMN_ORDER_MATCH, FILE_NOT_EMPTY_ROWS as WARN)
* `routing_policies`: priorities 10/20/30 (ERROR→Quarantine, UNIDENTIFIED→Quarantine, PASSED/WARNED→Intake)

### 10.3 Operational configuration

* **Validators:** `validationLevel: strict`, `validationAction: error` on all write-critical collections; changes only via the admin API (audited) — never ad-hoc `mongosh`.
* **TTL:** `event_log` PROCESSED-only, proposed 30 days (DEAD never expires; compliance confirmation open).
* **Capacity:** see HLD §5 growth table — ~47k small docs/day at 1,000 files/day + GridFS bytes; index budget reviewed at checkpoint R2.
* **Backup:** RPO ≤ 15 min (oplog/PITR), RTO ≤ 1 h — metadata + GridFS restored together (HLD §8).

## 11. LLD Review Checklist

1. Field completeness per collection vs. your screens/jobs — anything missing?
2. Enum values match operational vocabulary (statuses, severities)?
3. Denormalization choices OK (upload context on `file_registry` vs session-join)?
4. Seed data matches the real file types + intake volume paths?
5. Index set vs. expected query load — ops board filters adequate?